In [2]:
# Setup and imports.
import json
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

from sequential_decision_tree.evaluator import analyse_all_papers
from sequential_decision_tree.llm_client import MODEL

In [ ]:
# Configure input paths.
papers_dir = repo_root / "data" / "papers" / "json" / "1_to_100"
guidance_path = repo_root / "data" / "guidance_intro_md" / "guidance_intro.md"
checkpoint_path = papers_dir / ".sequential_decision_tree_checkpoint.json"

print(f"Papers dir: {papers_dir}")
print(f"Guidance exists: {guidance_path.exists()}")
print(f"Checkpoint path: {checkpoint_path}")

Papers dir: /home/user/Thesis/conservation-llm-critical-appraisal/data/papers/json/1_to_100
Guidance exists: True


In [4]:
import subprocess

#run cost check
result = subprocess.run(
    ["python", str(repo_root / "tests" / "api_credit_test.py")],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

{
  "data": {
    "label": "sk-or-v1-012...a08",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 1000,
    "limit_reset": null,
    "limit_remaining": 378.789733159,
    "include_byok_in_limit": false,
    "usage": 621.210266841,
    "usage_daily": 8.129475036,
    "usage_weekly": 427.73971868,
    "usage_monthly": 581.799520452,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": "2027-01-12T14:31:22.095Z",
    "creator_user_id": "user_35TUVIRsSsltNjDcienaMimoj8K",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}




In [ ]:
# Run sequential decision tree analysis for all papers (resume-capable).
results = analyse_all_papers(
    processed_dir=str(papers_dir),
    guidance_path=str(guidance_path),
    verbose=True,
    checkpoint_path=str(checkpoint_path),
    resume=True,
 )

print("Finished.")

--- Starting Analysis on All Papers in '/home/user/Thesis/conservation-llm-critical-appraisal/data/papers/json/1_to_100' ---
  -> Loaded CEECAT guidance from /home/user/Thesis/conservation-llm-critical-appraisal/data/guidance_intro_md/guidance_intro.md

--- Analyzing: 055 ---

--- Running Tree 1 ---

Node: q_1_1
Answer: yes

Node: q_1_2
Answer: no

Node: q_1_3
Answer: no

Final Result: HIGH RISK: Confounding not properly addressed.

--- Running Tree 2 ---

Node: q_2_1
Answer: seemingly yes

Node: q_2_2_a
Answer: no

Node: q_2_3_a
Answer: seemingly no

Final Result: LOW RISK

--- Running Tree 3 ---

Node: start
Answer: yes

Final Result: NOT APPLICABLE: This tree is only for observational studies.

--- Running Tree 4 ---

Node: start
Answer: yes

Node: q_4_1
Answer: yes

Node: q_4_2
Answer: seemingly no

Node: q_4_4_a
Answer: seemingly yes

Final Result: LOW

--- Running Tree 5 ---

Node: q_5_1
Answer: seemingly yes

Node: q_5_2_b
Answer: yes

Node: q_5_3_b
Answer: yes

Final Result: ME

AttributeError: 'NoneType' object has no attribute 'strip'

In [6]:
import subprocess

#run cost check
result = subprocess.run(
    ["python", str(repo_root / "tests" / "api_credit_test.py")],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

{
  "data": {
    "label": "sk-or-v1-012...a08",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 1000,
    "limit_reset": null,
    "limit_remaining": 373.44118082399996,
    "include_byok_in_limit": false,
    "usage": 626.558819176,
    "usage_daily": 13.478027371,
    "usage_weekly": 433.088271015,
    "usage_monthly": 587.148072787,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": "2027-01-12T14:31:22.095Z",
    "creator_user_id": "user_35TUVIRsSsltNjDcienaMimoj8K",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}




In [ ]:
# Save results with model suffix in filename.
output_dir = repo_root / "output" / "sequential_decision_tree_all_papers" / "1_to_100"
output_dir.mkdir(parents=True, exist_ok=True)
model_suffix = MODEL.replace("/", "_").replace(":", "_").replace(" ", "_")
output_path = output_dir / f"all_papers_{model_suffix}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved: {output_path}")

Saved: /home/user/Thesis/conservation-llm-critical-appraisal/output/sequential_decision_tree_all_papers/51_to_100/all_papers_anthropic_claude-opus-5.json


In [ ]:
# Quick preview.
paper_names = sorted(results.keys()) if isinstance(results, dict) else []
print(f"Total papers: {len(paper_names)}")
print("First papers:", paper_names[:5])

if paper_names:
    first = paper_names[0]
    print(f"\nSample result for {first}:")
    print(json.dumps(results[first], indent=2)[:1500])

Total papers: 48
First papers: ['051', '052', '053', '054', '055']

Sample result for 051:
{
  "1": {
    "result": "HIGH RISK: Confounding not properly addressed.",
    "path": [
      {
        "node": "q_1_1",
        "question": "1.1. Is it possible for the impact of the exposure or the effectiveness of the intervention to be confounded in this study?",
        "answer": "yes",
        "full_response": "**Evidence Extraction**\n\n- Study design and comparison: \"Barriers to tree establishment were compared for a grass- and shrub-dominated abandoned pasture (10 ha) and treefall gaps of a mature forest (225 ha) located 500 m from the pasture edge, at the same topographical position (Fig. 1).\"\n- Land-use history of the \"exposure\" site: \"The pasture was formed in 1969 when mature forest was cut, burned and planted with *Panicum maximum*\u2026 stocked at high densities\u2026 cleared with machetes and burned at least three times and was sprayed aerially once with 2,4-D\u2026\"\n- No